In [2]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

In [3]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LassoCV
from src.lasso_preprocessor import lasso_preprocessor
from src.encoder import preprocessor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [4]:
df = pd.read_csv("../data/train_cleaned.csv")
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [25]:
df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
df["TotalPorchSF"] = (
    df["OpenPorchSF"]
    + df["3SsnPorch"]
    + df["EnclosedPorch"]
    + df["ScreenPorch"]
    + df["WoodDeckSF"]
)

df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
df["IsRemodeled"] = (df["YearRemodAdd"] != df["YearBuilt"]).astype(int)

df["TotalBath"] = (
    df["FullBath"]
    + 0.5 * df["HalfBath"]
    + df["BsmtFullBath"]
    + 0.5 * df["BsmtHalfBath"]
)

df["QualTotalSF"] = df["OverallQual"] * df["TotalSF"]
df["QualGrLivArea"] = df["OverallQual"] * df["GrLivArea"]

df["HasGarage"] = (df["GarageArea"] > 0).astype(int)
df["HasBsmt"] = (df["TotalBsmtSF"] > 0).astype(int)
df["HasFireplace"] = (df["Fireplaces"] > 0).astype(int)
df["Has2ndFloor"] = (df["2ndFlrSF"] > 0).astype(int)

skewed = [
    "LotArea",
    "GrLivArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "GarageArea",
]

for col in skewed:
    df[col] = np.log1p(df[col])

In [26]:
X_train = df.drop(columns=["SalePrice"])
Y_train = np.log1p(df["SalePrice"])

In [40]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [51]:
lasso_model = LassoCV(cv=5, max_iter=20000, random_state=42)
xgb_model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    gamma=0.0,
    random_state=42,
    tree_method="hist",
    n_jobs=-1,
)

In [48]:
lasso_pipe = Pipeline([
    ("pre", lasso_preprocessor(X_train)),   # IMPORTANT: uses X to detect object cols
    ("lasso", lasso_model),
])

xgb_pipe = Pipeline([
    ("pre", preprocessor(X_train)),   # IMPORTANT: uses X to detect object cols
    ("xgb", xgb_model),
])

enet_pipe = Pipeline([
    ("pre", lasso_preprocessor(X_train)),   # IMPORTANT: uses X to detect object cols
    ("enet", enet_model),
])



In [39]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge,RidgeCV,ElasticNetCV

In [52]:
stack = StackingRegressor(
    estimators=[
        ("xgb", xgb_pipe),
        ("lasso", lasso_pipe)
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1,
)

# ---- 6) CV RMSE ----
scores = -cross_val_score(
    stack,
    X_train,
    Y_train,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)

print("RMSE mean:", scores.mean())
print("RMSE std :", scores.std())

# ---- 7) Fit on full train for final predictions (Kaggle submission) ----
# stack.fit(X_train, y_train)
# test_pred = stack.predict(X_test)
# If you trained on log1p(target), invert:
# test_pred = np.expm1(test_pred)

RMSE mean: 0.1184405848664927
RMSE std : 0.012547869109623125


Enet + XGboost + Lasso = 0.1295 +- 0.03225 , passthorugh = False
Enet + XGboost = 0.1291 += 0.030860, pass = False
Lasso + XGboost = 0.1291 +- 0.031257, pass= False
Lasso + Xgboost(5000 trees,0.01 learning rate) and MEta = Ridge(alpha = 0.1),cv = cv = 0.1287

Lasso + Xgboost(2000 trees,0.03 learning rate) and MEta = Ridge(alpha = 0.1),cv = 5 = 0.1184405 +- 0.01254

